# FD003 — two-stage optimization (internal)

Built from what the diagnostic run established:

| finding | consequence for this notebook |
|---|---|
| validation tracks test well (rho 0.90-0.92) but is **flat at the optimum** — trials 23 and 27 differ by 0.008 in validation and 1.35 in test | objective is **mean(val_exh, val_tl)**, which had zero regret in the bake-off |
| **winner's curse**: the selected trial drifted +0.80 on fresh seeds vs +0.27 for the baseline; 2-seed SE is ~0.23 and argmin over 30 costs ~0.48 optimism | **two-stage selection** — shortlist cheaply, then re-decide on many fresh seeds |
| `lr` pinned at the 0.005 ceiling in 8 of the top 10, `dropout` pinned at 0 | **widened space**, and batch size is now searched |
| batch 512 gives 31 steps/epoch, so 60 epochs is only **1,860 updates** — the model is under-trained, which is exactly why it wanted a bigger `lr` and no dropout | **120 epochs, patience 20**, and smaller batches available |

**Stage 1** — 40 Bayesian trials x 2 seeds, scored on the blended objective. Cheap and noisy; its
only job is to produce a shortlist.
**Stage 2** — the top 8 configurations re-evaluated on **10 fresh seeds each**. The decision is made
here, on low-noise estimates, which is what removes the winner's curse.
**Stage 3** — optional local refinement: a tightened Bayesian search in a box around the Stage 2
winner, then the same many-seed re-evaluation.
**Decision rule** — a configuration is adopted only if it beats the baseline by more than the
combined seed noise. Otherwise the result is a tie and the baseline stands.

Test is recorded throughout for diagnosis and never enters any selection.

Architecture and preprocessing fixed to the factor-grid winner: our parallel CNN-LSTM on Peng's
preprocessing. Run from the folder containing `train_FD003.txt`.

In [1]:
import os, json, gc, time, random, math
os.environ['PYTHONHASHSEED']='42'
import numpy as np, pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_squared_error
from scipy.stats import spearmanr
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Concatenate, Flatten, GlobalAveragePooling1D)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import keras_tuner as kt
def set_seeds(s):
    os.environ['PYTHONHASHSEED']=str(s); random.seed(s); np.random.seed(s)
    tf.keras.utils.set_random_seed(s)
def gpu_preflight():
    if not tf.config.list_physical_devices('GPU'): print('CPU mode'); return False
    try:
        x=np.random.rand(8,10,3).astype('float32'); y=np.random.rand(8,1).astype('float32')
        i=Input(shape=(10,3)); o=Dense(1)(GlobalAveragePooling1D()(Conv1D(4,3,padding='same')(i)))
        m=Model(i,o); m.compile('adam','mse'); m.fit(x,y,epochs=1,verbose=0); K.clear_session()
        print('GPU preflight passed'); return True
    except Exception as e:
        print('GPU preflight FAILED (%s)'%type(e).__name__); K.clear_session(); return False
USE_GPU=gpu_preflight(); DEVICE='/GPU:0' if USE_GPU else '/CPU:0'
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| device',DEVICE)

GPU preflight passed
TF 2.10.0 | device /GPU:0


In [2]:
SEED=42; RUL_CAP=125; W=50
EPOCHS, PATIENCE = 120, 20          # was 60/15 - the model was under-trained
STAGE1_TRIALS, STAGE1_SEEDS = 40, 2
SHORTLIST, STAGE2_SEEDS      = 8, 10
REFINE, REFINE_TRIALS        = True, 15
TESTLIKE_PER_UNIT, TESTLIKE_SEED = 50, 12345
set_seeds(SEED)

idx=['unit_nr','time_cycles']; setn=['setting_1','setting_2','setting_3']
sn=[f's_{i}' for i in range(1,22)]; cols=idx+setn+sn
train=pd.read_csv('train_FD003.txt',sep=r'\s+',header=None,names=cols)
test =pd.read_csv('test_FD003.txt', sep=r'\s+',header=None,names=cols)
y_test=pd.read_csv('RUL_FD003.txt',sep=r'\s+',header=None,names=['RUL'])
_g=GroupShuffleSplit(n_splits=1,train_size=0.8,random_state=SEED)
_ti,_vi=next(_g.split(train,groups=train.unit_nr))
TRAIN_UNITS=np.sort(train.iloc[_ti].unit_nr.unique())
VAL_UNITS  =np.sort(train.iloc[_vi].unit_nr.unique())
assert not set(TRAIN_UNITS)&set(VAL_UNITS)
feats=setn+sn+['time_cycles']
tr=train.copy(); mx=tr.groupby('unit_nr')['time_cycles'].transform('max')
tr['RUL']=(mx-tr['time_cycles']).clip(upper=RUL_CAP); te=test.copy()
sc=MinMaxScaler().fit(tr[tr.unit_nr.isin(TRAIN_UNITS)][feats])
tr[feats]=np.nan_to_num(sc.transform(tr[feats])); te[feats]=np.nan_to_num(sc.transform(te[feats]))
def win_ex(df,us):
    X,y=[],[]
    for u in us:
        d=df[df.unit_nr==u]; A=d[feats].to_numpy(np.float32); L=d['RUL'].to_numpy(np.float32)
        for e in range(W,len(d)+1): X.append(A[e-W:e]); y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(y,np.float32).reshape(-1,1)
def win_tl(df,us,rng,per):
    X,y=[],[]
    for u in us:
        d=df[df.unit_nr==u]; A=d[feats].to_numpy(np.float32); L=d['RUL'].to_numpy(np.float32)
        n=len(d)
        if n<W: continue
        for cut in rng.integers(W,n+1,size=per): X.append(A[cut-W:cut]); y.append(L[cut-1])
    return np.asarray(X,np.float32),np.asarray(y,np.float32).reshape(-1,1)
Xtr,ytr=win_ex(tr,TRAIN_UNITS); Xve,yve=win_ex(tr,VAL_UNITS)
Xvt,yvt=win_tl(tr,VAL_UNITS,np.random.default_rng(TESTLIKE_SEED),TESTLIKE_PER_UNIT)
Xte=[]
for u in te.unit_nr.unique():
    A=te[te.unit_nr==u][feats].to_numpy(np.float32)
    Xte.append(A[-W:] if len(A)>=W else np.vstack([np.repeat(A[:1],W-len(A),axis=0),A]))
Xte=np.asarray(Xte,np.float32)
yte=np.clip(y_test['RUL'].to_numpy(np.float32),0,RUL_CAP).reshape(-1,1)
print('train %s | val_exh %s | val_tl %s | test %s'%(Xtr.shape,Xve.shape,Xvt.shape,Xte.shape))
for b in (128,256,512):
    print('  batch %3d -> %4d steps/epoch, %6d updates at %d epochs'
          %(b,-(-len(Xtr)//b),-(-len(Xtr)//b)*EPOCHS,EPOCHS))

train (16092, 50, 25) | val_exh (3728, 50, 25) | val_tl (1000, 50, 25) | test (100, 50, 25)
  batch 128 ->  126 steps/epoch,  15120 updates at 120 epochs
  batch 256 ->   63 steps/epoch,   7560 updates at 120 epochs
  batch 512 ->   32 steps/epoch,   3840 updates at 120 epochs


In [3]:
def build(filters=128,lstm_units=128,dense=128,merged=256,dropout=0.2,l2reg=0.01,
          lr=1e-3,seed=SEED):
    kw=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed))
    inp=Input(shape=(W,len(feats)))
    c=Conv1D(filters,3,padding='same',activation='relu',**kw)(inp)
    c=MaxPooling1D(2,padding='same')(c); c=Flatten()(c)
    c=Dense(dense,activation='relu',kernel_regularizer=regularizers.l2(l2reg),**kw)(c)
    c=Dropout(dropout,seed=seed)(c)
    r=LSTM(lstm_units,return_sequences=True,**kw)(inp)
    r=LSTM(lstm_units,activation='tanh',**kw)(r)
    r=Dense(dense,activation='relu',kernel_regularizer=regularizers.l2(l2reg),**kw)(r)
    r=Dropout(dropout,seed=seed)(r)
    h=Concatenate()([c,r])
    h=Dense(merged,activation='relu',kernel_regularizer=regularizers.l2(l2reg),**kw)(h)
    h=Dropout(dropout,seed=seed)(h)
    m=Model(inp,Dense(1,activation='linear',**kw)(h))
    m.compile(optimizer=tf.keras.optimizers.Adam(lr),loss='mse'); return m

def evaluate(kw,seed):
    kw=dict(kw); batch=int(kw.pop('batch',512))
    K.clear_session(); gc.collect(); set_seeds(seed); t0=time.time()
    with tf.device(DEVICE):
        m=build(seed=seed,**kw)
        h=m.fit(Xtr,ytr,validation_data=(Xve,yve),epochs=EPOCHS,batch_size=batch,verbose=0,
                callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                           ReduceLROnPlateau('val_loss',factor=0.5,patience=8,min_lr=1e-6)])
        rm=lambda X,y: float(np.sqrt(mean_squared_error(y,m.predict(X,batch_size=1024,verbose=0))))
        out=dict(val_exh=rm(Xve,yve),val_tl=rm(Xvt,yvt),test=rm(Xte,yte),
                 params=int(m.count_params()),epochs_run=len(h.history['loss']),
                 secs=round(time.time()-t0,1))
    out['obj']=0.5*(out['val_exh']+out['val_tl'])          # the blended objective
    return out

def agg(kw,n_seeds,seed0):
    rs=[evaluate(kw,seed0+s) for s in range(n_seeds)]
    a={k:float(np.mean([r[k] for r in rs])) for k in ('val_exh','val_tl','test','obj','epochs_run','secs')}
    a['test_std']=float(np.std([r['test'] for r in rs],ddof=1)) if n_seeds>1 else 0.0
    a['obj_std'] =float(np.std([r['obj']  for r in rs],ddof=1)) if n_seeds>1 else 0.0
    a['test_best']=float(np.min([r['test'] for r in rs])); a['params']=rs[0]['params']
    a['n_seeds']=n_seeds; return a

DEFAULTS=dict(filters=128,lstm_units=128,dense=128,merged=256,dropout=0.2,l2reg=0.01,
              lr=1e-3,batch=512)
print('under-training check at the new budget (%d epochs, patience %d):'%(EPOCHS,PATIENCE))
b=agg(DEFAULTS,2,SEED)
print('  baseline: obj %.3f | val_exh %.3f | val_tl %.3f | test %.3f | epochs run %.0f/%d | %.0f s/run'
      %(b['obj'],b['val_exh'],b['val_tl'],b['test'],b['epochs_run'],EPOCHS,b['secs']))
print('  -> if epochs run is at the ceiling, 60 epochs really was cutting training short')

under-training check at the new budget (120 epochs, patience 20):
  baseline: obj 8.919 | val_exh 8.563 | val_tl 9.276 | test 10.516 | epochs run 103/120 | 39 s/run
  -> if epochs run is at the ceiling, 60 epochs really was cutting training short


## Stage 1 — cheap shortlist

In [4]:
SPACE=[('filters',32,384,'int'),('lstm_units',32,256,'int'),('dense',32,384,'int'),
       ('merged',64,512,'int'),('dropout',0.0,0.5,'float'),
       ('lr',1e-4,2e-2,'log'),('l2reg',1e-8,1e-1,'log'),('batch',0,2,'batch')]
BATCHES=[128,256,512]
def hp_kw(hp,box=None):
    kw={}
    for n,lo,hi,k in SPACE:
        if box and n in box: lo,hi=box[n]
        if k=='batch':  kw['batch']=BATCHES[hp.Int('batch_idx',int(lo),int(hi))]
        elif k=='int':  kw[n]=hp.Int(n,int(lo),int(hi),step=max(1,int((hi-lo)//20)))
        elif k=='log':  kw[n]=hp.Float(n,lo,hi,sampling='log')
        else:           kw[n]=hp.Float(n,lo,hi,step=0.05)
    return kw

S1=[]
class Stage1(kt.BayesianOptimization):
    def run_trial(self,trial,*a,**k):
        kw=hp_kw(trial.hyperparameters)
        try: r=agg(kw,STAGE1_SEEDS,SEED)
        except Exception as e:
            print('   trial %s FAILED (%s)'%(trial.trial_id,type(e).__name__))
            r=dict(val_exh=1e3,val_tl=1e3,test=1e3,obj=1e3,params=0,epochs_run=0,secs=0,
                   test_std=0,obj_std=0,test_best=1e3,n_seeds=STAGE1_SEEDS)
        S1.append(dict(trial=int(trial.trial_id),**kw,**r))
        print('   t%-3s obj %6.3f | val_exh %6.3f val_tl %6.3f | test %6.3f | ep %3.0f | %4.0fs'
              %(trial.trial_id,r['obj'],r['val_exh'],r['val_tl'],r['test'],r['epochs_run'],r['secs']))
        return {'obj':r['obj']}

t0=time.time()
tuner=Stage1(hypermodel=lambda hp: build(**{k:v for k,v in hp_kw(hp).items() if k!='batch'}),
             objective=kt.Objective('obj',direction='min'),
             max_trials=STAGE1_TRIALS,seed=SEED,
             directory='opt2stage_fd003',project_name='stage1',overwrite=True)
tuner.search()
T1=pd.DataFrame(S1); T1.to_csv('stage1_trials_fd003.csv',index=False)
pd.set_option('display.width',230)
print('\nstage 1: %d trials in %.0f s'%(len(T1),time.time()-t0))
r,_=spearmanr(T1['obj'],T1['test'])
print('objective vs test across stage 1: Spearman %+.3f'%r)
print('\nshortlist (top %d by objective):'%SHORTLIST)
SL=T1.nsmallest(SHORTLIST,'obj')
print(SL[['trial','obj','val_exh','val_tl','test','params','epochs_run','batch','lr','dropout']].to_string(index=False))

Trial 40 Complete [00h 02m 03s]
obj: 10.030312299728394

Best obj So Far: 8.540950536727905
Total elapsed time: 01h 10m 48s

stage 1: 40 trials in 4248 s
objective vs test across stage 1: Spearman +0.505

shortlist (top 8 by objective):
 trial      obj  val_exh   val_tl      test  params  epochs_run  batch       lr  dropout
    29 8.540951 8.148921 8.932980 10.901629 1107421        51.5    128 0.000599     0.00
    25 8.549481 8.142112 8.956851 11.363886 1851004        37.5    128 0.001782     0.05
    32 8.562452 8.236070 8.888834 11.074582 1772699        29.5    128 0.003220     0.00
    19 8.631398 8.206627 9.056169 10.836491 2592632        49.5    128 0.000620     0.00
    27 8.717777 8.286111 9.149444 10.814027 1842937        41.0    256 0.002151     0.00
    26 8.718161 8.279922 9.156401 11.161906 1857280        75.0    256 0.001083     0.10
    33 8.718986 8.384734 9.053238 10.938787 1917981        29.5    128 0.002246     0.00
    28 8.736913 8.286590 9.187237 11.835959 1102321

## Stage 2 — re-decide on fresh seeds

This is the step that removes the winner's curse: the shortlist is re-scored on 10 seeds none of
them have seen, and the decision is made on those estimates.

In [7]:
HPN=[n for n,_,_,_ in SPACE if n!='batch']+['batch']
def kw_of(row):
    get = (lambda n: getattr(row, n)) if hasattr(row, '_fields') else (lambda n: row[n])
    return {n: (int(get(n)) if n in ('filters','lstm_units','dense','merged','batch')
                else float(get(n))) for n in HPN}

In [9]:
HPN=[n for n,_,_,_ in SPACE if n!='batch']+['batch']
def kw_of(row):
    get = (lambda n: getattr(row, n)) if hasattr(row, '_fields') else (lambda n: row[n])
    return {n: (int(get(n)) if n in ('filters','lstm_units','dense','merged','batch')
                else float(get(n))) for n in HPN}
S2=[]
print('re-evaluating %d configurations on %d fresh seeds each...'%(len(SL)+1,STAGE2_SEEDS))
for nm,kw in [('baseline',DEFAULTS)]+[('trial %d'%r.trial,kw_of(r)) for r in SL.itertuples()]:
    a=agg(kw,STAGE2_SEEDS,SEED+5000)
    S2.append(dict(name=nm,**a,**{k:v for k,v in kw.items()}))
    print('  %-10s obj %6.3f +/- %5.3f | test %6.3f +/- %5.3f (best %6.3f)'
          %(nm,a['obj'],a['obj_std'],a['test'],a['test_std'],a['test_best']))
T2=pd.DataFrame(S2); T2.to_csv('stage2_shortlist_fd003.csv',index=False)
base=T2[T2.name=='baseline'].iloc[0]
win =T2.nsmallest(1,'obj').iloc[0]
print('\nbest on the blended objective: %s'%win['name'])
d=win['test']-base['test']; noise=math.sqrt(win['test_std']**2+base['test_std']**2)
print('  its test %.3f vs baseline %.3f -> %+.3f against combined noise %.3f'
      %(win['test'],base['test'],d,noise))
print('  verdict: %s'%('REAL improvement' if d<-noise else
                       'REAL regression' if d>noise else 'tie - keep the baseline'))
print('\noptimism removed by stage 2:')
for r in SL.itertuples():
    s2=T2[T2.name=='trial %d'%r.trial]
    if len(s2): print('  trial %-3d stage1 test %6.3f -> stage2 test %6.3f  (%+.3f)'
                      %(r.trial,r.test,s2.iloc[0]['test'],s2.iloc[0]['test']-r.test))

re-evaluating 9 configurations on 10 fresh seeds each...
  baseline   obj  9.159 +/- 0.390 | test 10.765 +/- 0.378 (best 10.182)
  trial 29   obj  9.086 +/- 0.227 | test 10.957 +/- 0.682 (best  9.705)
  trial 25   obj  8.668 +/- 0.160 | test 11.473 +/- 0.984 (best 10.438)
  trial 32   obj  8.803 +/- 0.220 | test 11.716 +/- 0.714 (best 10.432)
  trial 19   obj  8.774 +/- 0.300 | test 10.850 +/- 0.743 (best  9.709)
  trial 27   obj  8.841 +/- 0.179 | test 11.278 +/- 0.467 (best 10.321)
  trial 26   obj  8.740 +/- 0.218 | test 10.926 +/- 0.512 (best 10.477)
  trial 33   obj  8.794 +/- 0.312 | test 11.380 +/- 0.595 (best 10.426)
  trial 28   obj  8.847 +/- 0.258 | test 11.581 +/- 0.715 (best 10.539)

best on the blended objective: trial 25
  its test 11.473 vs baseline 10.765 -> +0.708 against combined noise 1.054
  verdict: tie - keep the baseline

optimism removed by stage 2:
  trial 29  stage1 test 10.902 -> stage2 test 10.957  (+0.055)
  trial 25  stage1 test 11.364 -> stage2 test 11.4

## Stage 3 — local refinement around the Stage 2 winner

In [10]:
if REFINE:
    seed_cfg=kw_of(win) if win['name']!='baseline' else DEFAULTS
    box={}
    for n,lo,hi,k in SPACE:
        if k=='batch': continue
        v=seed_cfg[n]
        if k=='log': box[n]=(max(lo,v/3.0),min(hi,v*3.0))
        elif k=='int': box[n]=(max(lo,int(v*0.6)),min(hi,int(v*1.6)+1))
        else: box[n]=(max(lo,v-0.15),min(hi,v+0.15))
    print('refining around %s:'%win['name'])
    for n,(a,b_) in box.items(): print('   %-11s %.6g .. %.6g'%(n,a,b_))
    S3=[]
    class Stage3(kt.BayesianOptimization):
        def run_trial(self,trial,*a,**k):
            kw=hp_kw(trial.hyperparameters,box=box)
            try: r=agg(kw,STAGE1_SEEDS,SEED)
            except Exception as e:
                r=dict(val_exh=1e3,val_tl=1e3,test=1e3,obj=1e3,params=0,epochs_run=0,secs=0,
                       test_std=0,obj_std=0,test_best=1e3,n_seeds=STAGE1_SEEDS)
            S3.append(dict(trial=int(trial.trial_id),**kw,**r))
            print('   r%-3s obj %6.3f | test %6.3f'%(trial.trial_id,r['obj'],r['test']))
            return {'obj':r['obj']}
    tuner3=Stage3(hypermodel=lambda hp: build(**{k:v for k,v in hp_kw(hp,box=box).items() if k!='batch'}),
                  objective=kt.Objective('obj',direction='min'),
                  max_trials=REFINE_TRIALS,seed=SEED,
                  directory='opt2stage_fd003',project_name='stage3',overwrite=True)
    tuner3.search()
    T3=pd.DataFrame(S3); T3.to_csv('stage3_refine_fd003.csv',index=False)
    top3=T3.nsmallest(3,'obj')
    print('\nre-evaluating the top 3 refinements on %d fresh seeds:'%STAGE2_SEEDS)
    S3F=[]
    for r in top3.itertuples():
        a=agg(kw_of(r),STAGE2_SEEDS,SEED+5000)
        S3F.append(dict(name='refine %d'%r.trial,**a)); 
        print('  refine %-3d obj %6.3f +/- %5.3f | test %6.3f +/- %5.3f'
              %(r.trial,a['obj'],a['obj_std'],a['test'],a['test_std']))
    T2=pd.concat([T2,pd.DataFrame(S3F)],ignore_index=True)
else:
    print('refinement skipped')

Trial 15 Complete [00h 01m 30s]
obj: 8.43702745437622

Best obj So Far: 8.383788108825684
Total elapsed time: 00h 22m 30s

re-evaluating the top 3 refinements on 10 fresh seeds:
  refine 5   obj  8.660 +/- 0.182 | test 11.056 +/- 0.512
  refine 14  obj  8.672 +/- 0.273 | test 11.326 +/- 0.627
  refine 1   obj  8.659 +/- 0.368 | test 11.098 +/- 0.678


## Final decision

In [11]:
base=T2[T2.name=='baseline'].iloc[0]
fin =T2.nsmallest(1,'obj').iloc[0]
d=fin['test']-base['test']; noise=math.sqrt(fin['test_std']**2+base['test_std']**2)
adopt = d < -noise
print('='*74)
print('%-22s obj %6.3f | test %6.3f +/- %5.3f (best run %6.3f)'
      %('baseline',base['obj'],base['test'],base['test_std'],base['test_best']))
print('%-22s obj %6.3f | test %6.3f +/- %5.3f (best run %6.3f)'
      %(fin['name'],fin['obj'],fin['test'],fin['test_std'],fin['test_best']))
print('difference %+.3f vs combined noise %.3f'%(d,noise))
print('DECISION: %s'%('adopt %s'%fin['name'] if adopt else 'tie - keep the baseline'))
print('='*74)
print('\nall candidates, decided on the blended objective (test shown for diagnosis only):')
print(T2[['name','obj','obj_std','val_exh','val_tl','test','test_std','test_best','params']]
      .sort_values('obj').to_string(index=False))
json.dump(dict(stage1=S1,stage2=T2.to_dict('records'),adopt=bool(adopt),
               winner=str(fin['name']),baseline_test=float(base['test']),
               winner_test=float(fin['test']),noise=float(noise)),
          open('two_stage_fd003.json','w'),indent=2,default=str)
print('\nsaved stage1_trials_fd003.csv, stage2_shortlist_fd003.csv, two_stage_fd003.json')

baseline               obj  9.159 | test 10.765 +/- 0.378 (best run 10.182)
refine 1               obj  8.659 | test 11.098 +/- 0.678 (best run  9.927)
difference +0.333 vs combined noise 0.776
DECISION: tie - keep the baseline

all candidates, decided on the blended objective (test shown for diagnosis only):
     name      obj  obj_std  val_exh   val_tl      test  test_std  test_best  params
 refine 1 8.658810 0.367703 8.253037 9.064582 11.097515  0.677945   9.926625 2830289
 refine 5 8.660262 0.182238 8.285879 9.034646 11.055904  0.511696  10.329413 1868952
 trial 25 8.667832 0.160280 8.284975 9.050689 11.472773  0.984435  10.438388 1851004
refine 14 8.671981 0.273207 8.324463 9.019499 11.325629  0.626551  10.556452 1751003
 trial 26 8.740189 0.218470 8.358457 9.121921 10.925713  0.511821  10.477046 1857280
 trial 19 8.774186 0.300389 8.339002 9.209370 10.849945  0.742565   9.709352 2592632
 trial 33 8.794140 0.312432 8.381905 9.206375 11.379911  0.595302  10.426411 1917981
 trial 32